<a href="https://colab.research.google.com/github/Suim00/hdac-knhanes-skipbreakfast-obesity/blob/main/06_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 보건의료 데이터분석 심화 — 7주차 실습
## 복합표본 회귀분석 — DAG와 교란 통제

---

**실습 자료** — 4주차·5주차와 같은 자료를 씁니다.
`knhanes_like_week4.csv` (n = 5,367 · 263개 조사구 · 18개 층) · `codebook_week4.csv`

| 파트 | 내용 |
|---|---|
| **실습 ①** | 인과구조 → 보정집합 → 다섯 모형 비교 → 단계적 선택법 재현 |
| **실습 ②** | 설계 기반 표준오차 → 계수의 설계효과 → 교호작용 → Table 2 |

> 오늘 자료에는 **모집단 참값**이 함께 제공됩니다.
> 여러분의 추정치가 맞았는지 **채점할 수 있습니다.** 실제 연구에서는 불가능한 일입니다.

---

### 이 노트북 사용법

- `# TODO` 가 붙은 셀이 여러분이 채울 곳입니다. **`____`(밑줄 네 개)가 빈칸**입니다.
- 빈칸을 채우지 않고 실행하면 `NameError: name '____' is not defined` 가 납니다. **정상입니다.** 채우면 사라집니다.
- **위에서부터 순서대로** 채우며 실행하십시오. 1-2의 `svyglm`이 이후 모든 셀의 기반이므로,
  **1-2 검증 출력의 두 줄이 같아진 뒤에** 다음으로 넘어가십시오.
- `____` 한 줄만 있는 셀은 코드 전체를 직접 작성하는 셀입니다.
- 1-3에서 만드는 `EST`에 모형 결과를 저장해 두면 2-2·2-5에서 다시 적합할 필요가 없습니다.
- 💬 표시가 있는 칸에는 직접 문장을 적으십시오.
  **오늘은 코드보다 이 칸이 중요합니다.** 오늘 배우는 것은 계산이 아니라 판단입니다.
- 제출 전에는 `____`가 하나도 남아 있지 않아야 `런타임 → 모두 실행`이 오류 없이 끝납니다.

### 오늘의 한 문장

> **변수 선택은 통계가 아니라 인과구조가 결정한다.**

p값도, 모형 적합도도, AIC도 기준이 아닙니다. 전부 자료에서 계산되는 양이기 때문입니다.
자료는 무엇이 원인이고 무엇이 결과인지 알려주지 않습니다.

### 오늘 확인할 것

같은 자료, 같은 노출(비만), 같은 결과(고혈압)로 다섯 개 모형을 돌립니다.
**참값은 1.687 하나뿐인데 추정치는 1.06에서 1.91까지 흩어집니다.**
어느 것이 맞는지는 자료가 알려주지 않습니다. 인과구조가 알려줍니다.

---
# 0. 환경 설정

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF   # 2-4에서 사용

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [ ]:
# Colab에서는 아래 두 줄의 주석(#)을 지우고 실행해 CSV 두 개를 올리십시오 (한 번만)
from google.colab import files
files.upload()

DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy(float)
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

Saving knhanes_like_week4.csv to knhanes_like_week4.csv
Saving codebook_week4.csv to codebook_week4.csv
n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


## 0-2. 4주차 함수 — 그대로 가져다 씁니다

수정하지 말고 실행만 하십시오. 오늘 이 함수의 **행렬 버전**을 만들 것입니다.

In [ ]:
def _var_psu(z, strata, psu):
    """조사구 단위로 합산한 뒤 층 안에서 분산을 구한다 (궁극집락 방법)"""
    t = pd.DataFrame({'st': strata, 'cl': psu, 'z': z})
    zc = t.groupby(['st', 'cl'])['z'].sum().reset_index()
    var = 0.0
    for s, g in zc.groupby('st'):
        m = len(g)
        if m > 1:
            var += m / (m - 1) * np.sum((g['z'] - g['z'].mean()) ** 2)
    return var


def svymean(y, w, strata, psu):
    """가중 평균/비율 + 설계 기반 표준오차 (4주차)"""
    y, w = np.asarray(y, float), np.asarray(w, float)
    ok = ~np.isnan(y)
    y, w = y[ok], w[ok]
    st, cl = np.asarray(strata)[ok], np.asarray(psu)[ok]
    th = np.sum(w * y) / np.sum(w)
    return th, np.sqrt(_var_psu(w * (y - th) / np.sum(w), st, cl))


print('4주차 함수 준비 완료')

4주차 함수 준비 완료


## 0-3. 파생변수와 모집단 참값

`obes`, `male`, `smk_cur` 는 원자료에 없습니다. 직접 만듭니다.

`prog`(건강프로그램 참여)는 **실제 자료에 없는 교육용 변수**입니다.
비만인 사람과 고혈압인 사람이 각각 프로그램에 더 많이 참여하는 구조로 만들었습니다.
즉 **비만과 고혈압의 공통 결과 = 충돌부(collider)** 입니다.
난수 씨앗을 고정했으니 모든 사람이 같은 값을 얻습니다.

In [ ]:
df['obes']    = (df['bmi'] >= 25).astype(float)
df['male']    = (df['sex'] == 1).astype(float)
df['smk_cur'] = (df['smk'] == 1).astype(float)

# 충돌부 — 비만과 고혈압이 모두 참여 확률을 높인다 (교육용, 씨앗 고정)
_rng = np.random.default_rng(20260922)
_p = 1 / (1 + np.exp(-(-1.6 + 1.5 * df['obes'] + 1.5 * df['htn'])))
df['prog'] = (_rng.random(len(df)) < _p).astype(float)

print('프로그램 참여율 (비가중)')
print(f"  비만 O · 고혈압 O : {df.loc[(df.obes==1)&(df.htn==1),'prog'].mean()*100:5.1f}%")
print(f"  비만 X · 고혈압 X : {df.loc[(df.obes==0)&(df.htn==0),'prog'].mean()*100:5.1f}%")

# 강의자가 제공하는 모집단 참값 (실제 연구에서는 알 수 없는 값)
TRUTH = {'M1': 1.891, 'M2': 1.687, 'M3': 1.005, 'M4': 1.016, 'M5': 1.664}
print()
print('모집단 참값(목표 추정량) = OR', TRUTH['M2'])

프로그램 참여율 (비가중)
  비만 O · 고혈압 O :  78.7%
  비만 X · 고혈압 X :  16.8%

모집단 참값(목표 추정량) = OR 1.687


---
---
# 실습 ① — 인과구조로 변수를 고른다

> **연구 질문: 비만은 고혈압 위험을 얼마나 높이는가?**
>
> 노출 = `obes` (BMI ≥ 25) · 결과 = `htn` (고혈압)

## 1-1. 먼저 그림을 그린다 (코드 없음)

**코드를 치기 전에** 종이에 DAG를 그리십시오.

다음 변수들을 놓고 화살표를 그려 보십시오 —
`obes` · `htn` · `age` · `male` · `smk_cur` · `sbp` · `prog` · `height`

규칙은 셋뿐입니다.
① 변수를 네모로 · ② 원인 → 결과로 화살표 · ③ 순환 없음

age → obes          age → htn
male → obes         male → smk_cur       smk_cur → htn
obes → htn          ← 추정 대상 (총효과)
obes → sbp          sbp → htn            ← sbp는 매개
obes → prog         htn → prog           ← prog는 충돌부
(height: 연결 없음 — 무관)

💬 **각 변수를 네 유형 중 하나로 분류하십시오.** (근거를 한 줄씩)

| 변수 | 유형 (교란/매개/충돌부/무관) | 근거 |
|---|---|---|
| `age` | 교란 | 연령이 높을수록 비만이 많아지고 혈압도 오르기 때문에. |
| `male` | 교란 | 성별이 비만 여부와 흡연 습관을 모두 다르게 만들고, 흡연이 혈압에 영향을 주기 때문에. |
| `smk_cur` | 교란 | 성별을 통해 비만과 연결되고 혈압의 원인이기도 하기 때문에. |
| `sbp` | 매개 | 비만 때문에 혈압이 오르고 그 혈압이 고혈압을 만들기 때문에. |
| `prog` | 충돌부 | 비만인 사람도, 고혈압인 사람도 프로그램에 더 많이 참여하기 때문에. |
| `height` | 무관 | 신장은 비만의 원인도 고혈압의 원인도 아니기 때문에. |

💬 **보정집합(adjustment set)으로 무엇을 고르겠습니까?**

> 보정집합으로는 age, male, smk_cur을 고르겠습니다.

💬 **그 이유는?**

> 이 연구의 목적은 비만이 고혈압에 미치는 총효과를 추정하는 것이므로, 비만에서 고혈압으로 가는 인과 경로는 모두 열어 두고 비만으로 들어오는 백도어만 닫아야 하기 때문입니다.

> 넣은 이유: age는 나이가 들수록 비만도 늘고 혈압도 오르기 때문에, male은 성별에 따라 비만과 흡연 양상이 다르고 흡연이 혈압에 영향을 주기 때문에, smk_cur는 성별을 통해 비만과 이어지면서 혈압의 원인이기도 하기 때문에 넣었습니다. 셋 다 비만 때문에 생긴 변수가 아니라 비만보다 먼저 있던 변수라서 보정해도 비만의 효과를 가리지 않습니다.

> 뺀 이유 : sbp는 비만 -> 고혈압 경로가 지나가는 매개변수라서 보정하면 총교과가 사라집니다. prog는 비만과 고혈압의 충돌부라서 보정하면 없던 연관이 생기기 때문입니다. height는 비만과 고혈압과는 무관하기 때문에 넣을 이유가 없습니다.

---
## 1-2. `svyglm` — 설계 기반 회귀

4주차 `svymean`을 행렬로 확장한 것입니다. 구조는 완전히 같습니다.

| 4주차 (평균) | 6주차 (회귀) |
|---|---|
| 잔차 `y − θ̂` | 점수 `w·(y − μ̂)·x` |
| 조사구별로 합산 | 조사구별로 합산 (벡터) |
| 층 안에서 분산 | 층 안에서 분산 (행렬) |
| SE = √분산 | V = A⁻¹ B A⁻¹ 의 대각 |

**샌드위치 추정량**
- 빵 `A` = Σ wᵢ · vᵢ · xᵢxᵢᵀ  (가중 정보행렬)
- 속 `B` = 점수의 궁극집락 분산
- `V = A⁻¹ B A⁻¹`

★ `freq_weights` 에는 **`w / w.mean()`** 을 넣습니다. 원값을 넣으면 표본이 45만 명이 됩니다.

In [ ]:
# TODO: 점수(score)의 궁극집락 분산행렬 B를 계산하는 함수를 완성하십시오.
#       힌트: _var_psu 와 구조가 같습니다. 스칼라 → 행렬로만 바뀝니다.

def _vcl(U, strata, psu):
    """점수행렬 U(n×p)를 조사구별로 합산한 뒤 층 안에서 분산행렬을 구한다"""
    t = pd.DataFrame(U); t['st'] = strata; t['cl'] = psu
    uc = t.groupby(['st', 'cl']).sum(numeric_only=True)   # 조사구별 점수합
    p = U.shape[1]
    B = np.zeros((p, p))
    for s, g in uc.groupby(level=0):
        m = len(g)                      # 층 안 조사구 수
        if m > 1:
            D = g.values - g.values.mean(axis=0)    # TODO: 조사구 점수합 − 층 평균
            B += m/(m-1) * D.T @ D             # TODO: m/(m-1) * D'D
    return B


def svyglm(y, X, w, strata, psu, family='binomial'):
    y = np.asarray(y, float); X = np.asarray(X, float); w = np.asarray(w, float)
    if family == 'binomial':
        m = sm.GLM(y, X, family=sm.families.Binomial(),
                   freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues
        v  = mu * (1 - mu)              # 이분형의 분산함수
    else:
        m = sm.GLM(y, X, family=sm.families.Gaussian(),
                   freq_weights=w / w.mean()).fit()
        mu = m.fittedvalues
        v  = np.ones_like(mu)           # 정규의 분산함수
    b = m.params
    A = (X * (w*v) [:, None]).T @ X         # TODO: X' diag(w*v) X
    U = (w*(y - mu))[:, None] * X         # TODO: 점수 w*(y-mu)*x  → n×p 행렬
    B = _vcl(U, strata, psu)
    Ai = np.linalg.inv(A)
    V  = Ai @ B @ Ai                    # 샌드위치
    return b, np.sqrt(np.diag(V)), m


# 검증 ① — 절편만 있는 로지스틱의 계수는 가중유병률의 로짓과 같다
X0 = np.ones((len(df), 1))
b0, se0, _ = svyglm(df['htn'], X0, w, ST, PS)
p_w = np.average(df['htn'], weights=w)
print(f"절편     = {b0[0]:.6f}   logit(가중유병률) = {np.log(p_w/(1-p_w)):.6f}")

# 검증 ② — 절편의 설계 기반 SE는 4주차 svymean의 SE를 델타법으로 바꾼 값과 같다
#   계수는 statsmodels가 계산하므로 ①만으로는 여러분이 채운 A·U·B를 검증할 수 없습니다.
#   ②가 맞아야 _vcl과 svyglm이 맞은 것입니다.
m_w, se_w = svymean(df['htn'], w, ST, PS)
print(f"SE(절편) = {se0[0]:.6f}   델타법 SE        = {se_w/(p_w*(1-p_w)):.6f}")
assert np.isclose(se0[0], se_w / (p_w * (1 - p_w))), 'SE가 다릅니다 — _vcl, A, U를 다시 확인하십시오'
print()
print('✅ svyglm 준비 완료 — 두 줄의 값이 같으면 다음으로 넘어가십시오')

절편     = -1.132045   logit(가중유병률) = -1.132045
SE(절편) = 0.041826   델타법 SE        = 0.041826

✅ svyglm 준비 완료 — 두 줄의 값이 같으면 다음으로 넘어가십시오


---
## 1-3. M1 무보정 → M2 보정집합 — 교란의 크기

먼저 보정 없이, 다음에 보정집합 {age, male, smk_cur}으로.

In [ ]:
# TODO: M1(무보정)과 M2(보정집합)를 적합하고 OR과 95% CI를 출력하십시오.
#       참값과 나란히 놓고 신뢰구간이 참값을 포함하는지 확인하십시오.

def or_ci(b, se, i=1):
    """i번째 계수의 OR과 95% CI"""
    return np.exp(b[i]), np.exp(b[i] - 1.96*se[i]), np.exp(b[i] + 1.96*se[i])        # TODO: exp(b), exp(b-1.96se), exp(b+1.96se)


EST = {}                           # 모형별 결과 저장소 — 1-4 ~ 1-6, 2-2, 2-5에서 다시 씁니다

for key, cols in [('M1', ['obes']),
                  ('M2', ['obes', 'age', 'male', 'smk_cur'])]:
    X = np.column_stack([np.ones(len(df)), df[cols].values])     # TODO: 절편 추가한 설계행렬
    b, se, m =  svyglm(df['htn'], X, w, ST, PS)     # TODO: svyglm 호출
    o, lo, hi = or_ci(b, se)
    EST[key] = {'cols': cols, 'b': b, 'se': se, 'model': m, 'or': (o, lo, hi)}
    print(f"{key}  OR = {o:.2f} ({lo:.2f}-{hi:.2f})  참값 1.69  포함: {lo <= 1.69 <= hi}")    # TODO: 모형 이름, OR (95% CI), 참값, 참값 포함 여부

M1  OR = 1.91 (1.68-2.17)  참값 1.69  포함: True
M2  OR = 1.69 (1.44-1.97)  참값 1.69  포함: True


---
## 1-4. M3 — 매개변수를 보정하면 (과보정 ①)

수축기혈압을 추가로 보정합니다. **"혈압을 통제했으니 더 정확하겠지"** 라고 기대하며.

In [ ]:
# TODO: M3(M2 + sbp)를 적합하십시오. M2와 비교하십시오.
#       1-3의 반복문을 그대로 쓰고 결과를 EST['M3']에 저장하십시오.
for key, cols in [('M3', ['obes', 'age', 'male', 'smk_cur', 'sbp'])]:
    X = np.column_stack([np.ones(len(df)), df[cols].values])
    b, se, m = svyglm(df['htn'], X, w, ST, PS)
    o, lo, hi = or_ci(b, se)
    EST[key] = {'cols': cols, 'b': b, 'se': se, 'model': m, 'or': (o, lo, hi)}
    print(f"{key}  OR = {o:.2f} ({lo:.2f}-{hi:.2f})  참값 1.69  포함: {lo <= 1.69 <= hi}")

print()
for key in ['M1', 'M2', 'M3']:
    o, lo, hi = EST[key]['or']
    print(f"{key}  OR = {o:.2f} ({lo:.2f}-{hi:.2f})   log OR = {EST[key]['b'][1]:.3f}   SE = {EST[key]['se'][1]:.3f}")

M3  OR = 1.14 (0.96-1.37)  참값 1.69  포함: False

M1  OR = 1.91 (1.68-2.17)   log OR = 0.648   SE = 0.065
M2  OR = 1.69 (1.44-1.97)   log OR = 0.522   SE = 0.080
M3  OR = 1.14 (0.96-1.37)   log OR = 0.135   SE = 0.092


---
## 1-5. M4 — 충돌부를 보정하면 (과보정 ②)

건강프로그램 참여(`prog`)를 추가합니다.
연구자의 논리: **"참여자와 비참여자는 건강행동이 다르니 통제해야 한다"** — 그럴듯합니다.

In [ ]:
# TODO: prog가 왜 충돌부인지 먼저 자료로 확인하고, M4를 적합하십시오.
#       확인할 것: (1) 비만 → prog (2) 고혈압 → prog
#       결과는 EST['M4']에 저장하십시오.

bp, sep, _ = svyglm(df['prog'], np.column_stack([np.ones(len(df)), df[['obes', 'htn']]]), w, ST, PS)
for i, nm in [(1, 'obes'), (2, 'htn')]:
    print(f"prog ~ {nm}: OR = {or_ci(bp, sep, i)[0]:.2f} ({or_ci(bp, sep, i)[1]:.2f}-{or_ci(bp, sep, i)[2]:.2f})")

cols = ['obes', 'age', 'male', 'smk_cur', 'prog']
b, se, m = svyglm(df['htn'], np.column_stack([np.ones(len(df)), df[cols].values]), w, ST, PS)
o, lo, hi = or_ci(b, se)
EST['M4'] = {'cols': cols, 'b': b, 'se': se, 'model': m, 'or': (o, lo, hi)}
print(f"M4  OR = {o:.2f} ({lo:.2f}-{hi:.2f})  참값 1.69  포함: {lo <= 1.69 <= hi}")

prog ~ obes: OR = 4.33 (3.74-5.02)
prog ~ htn: OR = 4.02 (3.51-4.61)
M4  OR = 1.06 (0.90-1.25)  참값 1.69  포함: False


---
## 1-6. M5 — 무관한 변수를 넣으면 (무해)

신장(`height`)을 넣습니다. 비만의 원인도, 고혈압의 원인도 아닙니다.

In [ ]:
# TODO: M5(M2 + height)를 적합하고 M2와 비교하십시오. 결과는 EST['M5']에 저장하십시오.
#       마지막에 다섯 모형(M1~M5)의 OR · 95% CI · 참값을 한 표로 정리하십시오.

cols = ['obes', 'age', 'male', 'smk_cur', 'height']
X = np.column_stack([np.ones(len(df)), df[cols].values])
b, se, m = svyglm(df['htn'], X, w, ST, PS)
EST['M5'] = {'cols': cols, 'b': b, 'se': se, 'model': m, 'or': or_ci(b, se)}

summary = pd.DataFrame(
    [(k, f"{EST[k]['or'][0]:.2f}", f"({EST[k]['or'][1]:.2f}-{EST[k]['or'][2]:.2f})", 1.69,
      'O' if EST[k]['or'][1] <= 1.69 <= EST[k]['or'][2] else 'X') for k in ['M1', 'M2', 'M3', 'M4', 'M5']],
    columns=['모형', 'OR', '95% CI', '참값', '포함'])
print(summary.to_string(index=False))

모형   OR      95% CI   참값 포함
M1 1.91 (1.68-2.17) 1.69  O
M2 1.69 (1.44-1.97) 1.69  O
M3 1.14 (0.96-1.37) 1.69  X
M4 1.06 (0.90-1.25) 1.69  X
M5 1.68 (1.43-1.97) 1.69  O


---
## 1-7. 단계적 선택법 — 통계에게 변수 선택을 맡기면 ★

후보 변수 16개를 주고 전진선택(p < 0.05)을 돌립니다.
`obes`는 관심 노출이므로 항상 남겨 둡니다 (연구자가 흔히 그렇게 합니다).

In [ ]:
# TODO: 전진 단계적 선택을 구현하십시오. (선택 과정은 속도를 위해 비가중 sm.Logit으로 합니다)
#       각 단계에서 (1) 가장 p값이 작은 후보를 추가하고
#                  (2) 그때 비만 OR이 어떻게 변하는지 출력하십시오.

CAND = ['age','male','smk_cur','sbp','dbp','prog','bmi','chol','glucose',
        'height','weight','edu','income','drink','exercise','urban']

dfc = df.dropna(subset=CAND + ['obes','htn']).copy()   # chol/glucose 결측 → listwise
print(f'완전자료 n = {len(dfc):,}  (원래 {len(df):,})')

sel, remain = ['obes'], [c for c in CAND]
y = dfc['htn'].astype(float)

while True:
    best = None
    for c in remain:
        f = sm.Logit(y, sm.add_constant(dfc[sel + [c]].astype(float))).fit(disp=0)
        pv = f.pvalues[c]                        # TODO: 후보 c의 p값
        if best is None or pv < best[1]:
            best = (c, pv)
    if best is None or best[1] >= 0.05:           # TODO: 중단 조건 (p >= 0.05)
        break
    sel.append(best[0]); remain.remove(best[0])
    f = sm.Logit(y, sm.add_constant(dfc[sel].astype(float))).fit(disp=0)
    print(f"추가: {best[0]:<9} p = {best[1]:.4g}  비만 OR = {np.exp(f.params['obes']):.2f}")                          # TODO: 추가된 변수, p값, 그때의 비만 OR

# TODO: 최종 선택된 변수(sel)로 svyglm을 다시 적합해 설계 기반 비만 OR과 95% CI를 구하고 참값과 비교하십시오.
#       (보고할 값은 이 설계 기반 추정치입니다)
print("최종 선택:", sel)
X = np.column_stack([np.ones(len(dfc)), dfc[sel].values])
b, se, m = svyglm(dfc['htn'], X, dfc['wt'].values, dfc['kstrata'].values, dfc['psu'].values)
o, lo, hi = or_ci(b, se)
print(f"단계적 선택 모형: 비만 OR = {o:.2f} ({lo:.2f}-{hi:.2f})  참값 1.69  포함: {lo <= 1.69 <= hi}")

완전자료 n = 4,484  (원래 5,367)
추가: sbp       p = 2.652e-215  비만 OR = 1.15
추가: prog      p = 2.974e-36  비만 OR = 0.71
추가: age       p = 8.534e-27  비만 OR = 0.71
추가: dbp       p = 8.028e-08  비만 OR = 0.72
최종 선택: ['obes', 'sbp', 'prog', 'age', 'dbp']
단계적 선택 모형: 비만 OR = 0.71 (0.57-0.89)  참값 1.69  포함: False


---
---
# 실습 ② — 설계, 교호작용, Table 2

## 2-1. 복합표본 선형회귀

결과가 연속형일 때. `family='gaussian'` 으로만 바꾸면 됩니다.

**BMI → 수축기혈압**에서 연령 교란의 크기를 눈으로 봅니다.

In [ ]:
# TODO: sbp ~ bmi 를 (1) 무보정 (2) age·male 보정 두 가지로 적합하십시오.
#       계수의 해석까지 한 줄 적으십시오.

for nm, cols in [('무보정', ['bmi']), ('age·male 보정', ['bmi', 'age', 'male'])]:
    X = np.column_stack([np.ones(len(df)), df[cols].values])
    b, se, m = svyglm(df['sbp'], X, w, ST, PS, family='gaussian')
    print(f"{nm:<12} bmi 계수 = {b[1]:.3f} mmHg  (SE {se[1]:.3f}, 95% CI {b[1]-1.96*se[1]:.3f} ~ {b[1]+1.96*se[1]:.3f})")

# 해석: 연령과 성별을 보정하면 BMI가 1 kg/m² 높을 때 수축기혈압이 평균 약 0.82 mmHg 높습니다.

무보정          bmi 계수 = 1.189 mmHg  (SE 0.061, 95% CI 1.069 ~ 1.309)
age·male 보정  bmi 계수 = 0.822 mmHg  (SE 0.063, 95% CI 0.700 ~ 0.945)


---
## 2-2. 계수의 설계효과 — 설계를 무시하면 얼마나 틀리는가

4주차에서 **평균**의 설계효과를 봤습니다 (고혈압 1.50, 수축기혈압 3.11).
**회귀계수**의 설계효과는 어떨까요?

In [ ]:
# TODO: M2 모형에서 계수별로 설계효과를 계산하십시오.
#       Deff = (설계 기반 SE / 모형 기본 SE)^2
#       설계 기반 SE는 EST['M2']['se'], 모형 기본 SE는 EST['M2']['model'].bse 에 있습니다
#       ※ '모형 기본 SE'는 GLM이 보고하는 값. 4주차의 '단순 SE'(SRS 공식)와는 기준이 다르다

d = EST['M2']
tab = pd.DataFrame({'설계 SE': d['se'], '모형 SE': d['model'].bse},
                   index=['절편'] + d['cols'])
tab['Deff'] = (tab['설계 SE'] / tab['모형 SE'])**2
print(tab.round(3))

         설계 SE  모형 SE   Deff
절편       0.177  0.154  1.324
obes     0.080  0.075  1.121
age      0.003  0.003  1.059
male     0.080  0.077  1.070
smk_cur  0.098  0.086  1.298


---
## 2-3. 교호작용 — 효과가 집단마다 다른가

"비만의 영향이 남녀에서 다른가?"

In [ ]:
# TODO: (1) obes × male 교호작용항을 넣어 검정하고
#       (2) 남녀 각각 층별 OR을 구해 비교하십시오.
#           자료를 잘라내지 말고 도메인 방식으로 — 가중치에 지시변수를 곱합니다 (5주차)

# (1) obes × male 교호작용항 검정
df['obes_male'] = df['obes'] * df['male']
cols = ['obes', 'male', 'obes_male', 'age', 'smk_cur']
X = np.column_stack([np.ones(len(df)), df[cols].values])
b, se, _ = svyglm(df['htn'], X, w, ST, PS)
z = b[3] / se[3]
print(f"교호작용 obes×male: 계수 = {b[3]:.3f}, SE = {se[3]:.3f}, z = {z:.2f}, p = {2*stats.norm.sf(abs(z)):.3f}")

# (2) 남녀 각각 층별 OR (도메인 방식: 가중치에 지시변수를 곱한다)
for g, nm in [(0, '여성'), (1, '남성')]:
    wd = w * (df['male'] == g).values
    X = np.column_stack([np.ones(len(df)), df[['obes', 'age', 'smk_cur']].values])
    b, se, _ = svyglm(df['htn'], X, wd, ST, PS)
    o, lo, hi = or_ci(b, se)
    print(f"{nm}: 비만 OR = {o:.2f} ({lo:.2f}-{hi:.2f})")

교호작용 obes×male: 계수 = -0.044, SE = 0.171, z = -0.26, p = 0.797
여성: 비만 OR = 1.72 (1.40-2.11)
남성: 비만 OR = 1.65 (1.28-2.12)


---
## 2-4. 다중공선성 — 같은 것을 두 번 넣으면

`bmi`, `height`, `weight` 를 동시에 넣어 봅니다. BMI = 체중 / 신장².

In [ ]:
# TODO: VIF를 계산하십시오. (0번 셀에서 불러온 VIF 함수 사용)
#       bmi · height · weight 를 연령 · 성별 · 흡연과 함께 넣으십시오.

cols = ['bmi', 'height', 'weight', 'age', 'male', 'smk_cur']
Xv = sm.add_constant(df[cols].astype(float))
vif = pd.Series([VIF(Xv.values, i) for i in range(1, Xv.shape[1])], index=cols)
print(vif.round(1))

bmi        97.1
height     47.8
weight     97.6
age         1.0
male        2.3
smk_cur     1.1
dtype: float64


---
## 2-5. Table 2 완성

논문의 두 번째 표.

제시 원칙
- 무보정 모형과 보정 모형을 **나란히** (보정으로 무엇이 달라졌는지 독자가 볼 수 있게)
- **관심 노출의 계수를 맨 위에**
- 보정변수의 계수는 실을 수는 있으나 **인과적으로 해석하지 않는다** (Table 2 fallacy)
- 각주에 모형·보정변수·설계 정보를 명시

In [ ]:
# TODO: Table 2를 만드십시오.
#       열: 무보정 OR (95% CI) | 보정 OR (95% CI)
#       행: 비만 + 보정변수들
#       비만의 무보정·보정 결과는 EST['M1'], EST['M2']에 이미 있습니다

fmt = lambda o, lo, hi: f"{o:.2f} ({lo:.2f}-{hi:.2f})"
d2 = EST['M2']
rows = []
for label, v in [('비만 (BMI ≥ 25)', 'obes'), ('연령 (1세 증가)', 'age'),
                 ('남성', 'male'), ('현재흡연', 'smk_cur')]:
    bu, seu, _ = svyglm(df['htn'], np.column_stack([np.ones(len(df)), df[[v]].values]), w, ST, PS)
    i = d2['cols'].index(v) + 1
    rows.append([label, fmt(*or_ci(bu, seu)), fmt(*or_ci(d2['b'], d2['se'], i))])
table2 = pd.DataFrame(rows, columns=['변수', '무보정 OR (95% CI)', '보정 OR (95% CI)'])
print(table2.to_string(index=False))

print("""
주) 결과변수 = 고혈압. 설계 기반 복합표본 로지스틱 회귀(n = 5,367). 보정 모형은 연령·성별·현재흡연을 함께 보정함.
    가중치로 계수를 추정하고, 층(18개)·집락(263개 조사구)을 반영한 샌드위치(Taylor 선형화) 표준오차로 95% 신뢰구간을 계산함.
    보정변수(연령·성별·현재흡연)의 계수는 인과적으로 해석할 수 없음 (Table 2 fallacy).
    이 값들은 비만의 효과를 추정하기 위해 구성한 모형에서 나온 것이기 때문임.""")

           변수  무보정 OR (95% CI)   보정 OR (95% CI)
비만 (BMI ≥ 25) 1.91 (1.68-2.17) 1.69 (1.44-1.97)
   연령 (1세 증가) 1.08 (1.07-1.08) 1.08 (1.07-1.08)
           남성 0.92 (0.81-1.05) 0.91 (0.78-1.06)
         현재흡연 1.22 (1.05-1.43) 1.59 (1.31-1.93)

주) 결과변수 = 고혈압. 설계 기반 복합표본 로지스틱 회귀(n = 5,367). 보정 모형은 연령·성별·현재흡연을 함께 보정함.
    가중치로 계수를 추정하고, 층(18개)·집락(263개 조사구)을 반영한 샌드위치(Taylor 선형화) 표준오차로 95% 신뢰구간을 계산함.
    보정변수(연령·성별·현재흡연)의 계수는 인과적으로 해석할 수 없음 (Table 2 fallacy).
    이 값들은 비만의 효과를 추정하기 위해 구성한 모형에서 나온 것이기 때문임.


---
## 2-6. Methods 문단 쓰기 💬

위에서 만든 표를 쓸 때 논문 방법론에 들어갈 문단을 직접 작성하십시오.
아래 네 요소를 **자기 숫자로** 채우십시오.

① 인과구조와 보정집합 선정 근거 · ② 회귀모형과 추정 방법
③ 복합표본 설계 반영 방법 · ④ 보정하지 않은 변수와 그 이유

💬 **Methods 문단**

> 비만(obes, BMI ≥ 25)이 고혈압(htn)에 미치는 총효과를 추정하기 위해 DAG를 그려 변수의 역할을 구분했다. age, male, smk_cur는 비만보다 먼저 있으면서 obes와 htn 양쪽에 영향을 주는 교란변수이므로 보정집합을 {age, male, smk_cur}로 정했다. 표본가중치(wt)를 반영한 로지스틱 회귀(svyglm)로 OR과 95% CI를 구했고, 층(kstrata, 18개)과 집락(psu, 263개 조사구)을 반영한 샌드위치(Taylor 선형화) 분산추정으로 표준오차를 계산했다(n = 5,367). obes의 무보정 OR(M1)은 1.91(1.68–2.17), 보정 OR(M2)은 1.69(1.44–1.97)였다. sbp는 obes → sbp → htn 경로의 매개변수라 보정하면 총효과가 사라지므로 제외했다. prog는 obes와 htn 양쪽의 결과(충돌부)라 보정하면 없던 연관이 생기므로 제외했다. height는 obes와 htn 어느 쪽의 원인도 아니어서 보정할 이유가 없어 제외했다.

💬 **오늘 다섯 모형 중 하나를 골라 논문에 보고한다면 어느 것이고, 왜입니까?**

> M2(obes + age, male, smk_cur)입니다. obes로 들어오는 백도어(age, male, smk_cur)만 닫고 obes → htn 경로는 열어 두어 총효과 추정 목적에 맞기 때문입니다. 선택 근거는 p값이 아니라 인과구조입니다. M3(sbp 추가, OR 1.14)와 M4(prog 추가, OR 1.06)는 과보정이고, M5(height 추가, OR 1.68)는 M2와 같지만 넣을 이유가 없습니다.

💬 **`prog`(프로그램 참여)를 보정한 논문을 심사한다고 생각해 보십시오.
저자에게 무엇을 물을 것입니까?** (두 문장)

> prog가 obes와 htn 양쪽의 결과(충돌부)가 아닌지, 그렇다면 prog를 보정해 obes–htn 사이에 없던 연관이 생기거나 효과가 왜곡되지 않았는지 DAG로 설명해 주십시오. prog를 뺀 모형의 obes OR도 함께 제시해 보정 전후 변화를 보여 주십시오.

---
---
# 제출

## 7주차 실습 + 개인 과제: 노트북 제출

- [V] 모든 `# TODO` 셀을 채우고 실행 (`____` 가 하나도 남지 않게)
- [V] 💬 칸 모두 작성 (특히 1-1 변수 분류표, 2-6 Methods 문단)
- [V] 아래 **개인 과제** 칸 ① ~ ⑤ 작성
- [ ] `런타임 → 모두 실행` 으로 처음부터 끝까지 오류 없이 실행되는지 확인
- [V] `파일 → .ipynb 다운로드`
- [V] 파일명 `7주차_학번_이름.ipynb`
- [ ] LMS 과제란에 **업로드** (GitHub 링크가 아니라 파일)

## 과제 (별도 안내문 참조)

**DAG 기반 보정집합 설계와 복합표본 회귀분석**

| 구분 | 항목 | 배점 |
|---|---|---|
| 개인 (60) | ① 노출·결과 확정 + DAG 작성 (총효과/직접효과 명시) | 20 |
|  | ② 후보변수 네 유형 분류 + 변수별 근거 | 15 |
|  | ③ 보정집합 결정과 선정 이유 | 10 |
|  | ④ 복합표본 회귀 + 설계 기반(샌드위치) SE — 무보정과 나란히 | 10 |
|  | ⑤ Table 2 + 각주 (Table 2 fallacy 경고 포함) | 5 |
| 팀 (40) | B-1 배정 노출 → 고혈압 DAG · 변수 분류 · 보정집합 | 15 |
|  | B-2 복합표본 로지스틱 회귀 — 무보정과 보정, 설계 기반 SE | 15 |
|  | B-3 매개변수 또는 충돌부를 일부러 넣은 모형 + 변화 설명 | 10 |

개인 60점 중 **①②가 35점**, ③까지 합치면 45점입니다. 코드가 아니라 판단에 점수가 있습니다.
팀 산출물은 이 노트북이 아니라 팀 저장소(`docs/06_dag.md`, `code/06_regression.ipynb`)에 커밋합니다.

마감: **10월 27일**

---
# 개인 과제 — 본인이 정한 노출–결과

실습의 '비만 → 고혈압'은 쓸 수 없습니다 (안내문 1절의 예시 중에서 고르거나 직접 정하십시오).
실습 1-2의 `svyglm`, 1-3의 `or_ci` 를 그대로 쓰면 됩니다.

## ① 노출·결과 확정 + DAG 작성 (20점)

💬 **노출 = 음주(drink)**  · **결과 = 수축기 혈압(sbp)**  · **추정 대상 (총효과 / 직접효과) = 총효과**

💬 **DAG** — 화살표 목록 (변수 13개, 순환 없음). `*` = 방향이 불확실한 화살표, `[ ]` = 자료에 없는(측정되지 않은) 변수

```
age → drink            age → sbp
male → drink           male → sbp            male → smk
edu → income           edu → drink           edu → sbp
income → drink         income → sbp
smk → sbp              smk → drink *         (smk = smk_cur · smk_former)
drink → sbp            ← 추정 대상 (총효과)
drink → bmi            bmi → sbp             ← bmi는 매개
drink → exercise *     exercise → sbp        ← exercise도 매개일 수 있음 (방향 불분명)
sbp → htn_med          income → htn_med      ← htn_med는 결과의 하류
[sbp_prev] → drink *   [sbp_prev] → sbp      ← 과거 혈압 (측정되지 않음)
[diet] → drink         [diet] → sbp          ← 식습관·염분 (측정되지 않음)
(height: 연결 없음 — 무관)
```

* 설명

**smk → drink \*** : 흡연과 음주 중 무엇이 먼저인지 단면자료로는 알 수 없어서 *를 붙였습니다. 반대로 drink → smk라면 흡연은 교란이 아니라 매개가 됩니다.

**drink → exercise \*** : 음주가 운동 습관을 바꾸는지, 운동하는 사람이 술을 덜 마시는지 알 수 없습니다. 반대 방향(exercise → drink)이면 운동은 교란입니다.

**[sbp_prev] → drink \*** : 단면자료라서 고혈압 진단 후 술을 줄이는 역방향이 가능합니다. 이를 `htn → drink`로 그리면 `drink → sbp → htn → drink`처럼 순환(A→B→A)이 되어 DAG가 아니므로, 시간 순서를 나누어 "과거 혈압이 현재 음주와 현재 혈압의 공통 원인"으로 표현했습니다. 과거 혈압은 자료에 없어 측정되지 않은 교란으로 남습니다.

**순환을 피하려고 뺀 것** : `htn → drink`(위 설명), `htn_med → sbp`(약이 혈압을 낮춤, `sbp → htn_med`와 함께 쓰면 순환), `sbp → htn`(htn은 sbp에서 만든 변수라 따로 그리지 않음). 약이 측정 혈압을 낮추는 효과는 한계로 적습니다.

## ② 후보변수 네 유형 분류 (15점)

| 변수 | 유형 (교란/매개/충돌부/무관) | 근거 (한 줄) |
|---|---|---|
| age | 교란 | 연령이 음주 양상과 혈압 모두에 영향을 줌 |
| male | 교란 | 성별이 음주 수준과 혈압 모두에 영향을 줌 (흡연을 통한 경로도 있음) |
| smk (smk_cur, smk_former) | 교란* | 흡연은 혈압의 원인이고 음주와 함께 나타나는 습관임. 방향이 불확실해(음주 → 흡연이면 매개) * 표시 |
| edu, income | 교란 | 사회경제적 수준이 음주와 혈압 모두에 영향을 줌 (edu → income 경로도 있음) |
| bmi | 매개 | 음주가 BMI를 바꾸고 BMI가 혈압을 바꾸는 경로 위에 있음 (음주 → bmi → 혈압, 방향은 아래 ④-2에서 자료로 확인) |
| exercise | 매개* | 음주가 운동 습관을 바꾸고 운동이 혈압을 바꾸는 경로로 가정함. 방향이 불분명해 민감도 분석에만 넣음 |
| htn_med | 충돌부 (결과의 하류) | 혈압(결과)이 복용의 원인이고 소득(의료이용)도 복용의 원인임 (sbp → htn_med ← income). 결과의 결과라서 보정하면 결과를 일부 통제하는 꼴이 되어 왜곡됨 |
| height | 무관 | 음주와 혈압 어느 쪽의 원인도 아님 |
| sbp_prev, diet (측정되지 않음) | 교란 (미측정) | 과거 혈압은 진단 후 절주를 통해 음주에, 식습관·염분은 음주와 혈압 양쪽에 영향을 줌. 자료에 없어 보정할 수 없음 |

## ③ 보정집합 결정과 이유 (10점)

💬 **보정집합 = { age, male, smk_cur, smk_former, edu, income }**

💬 **넣은 변수와 뺀 변수 각각의 이유** (한 문단 이상)

> 총효과를 추정하므로 음주로 들어오는 백도어만 닫고 음주 → 혈압 경로는 모두 열어 둡니다. 음주에서 화살표를 거꾸로 거슬러 혈압으로 가는 백도어는 연령·성별·교육·소득·흡연을 거치는 경로뿐이고(예: 음주 ← 연령 → 혈압, 음주 ← 흡연 → 혈압), 이 변수들은 모두 음주보다 앞서 음주와 혈압에 영향을 주기에 넣었습니다. 측정된 변수만 놓고 보면 이 다섯 변수(흡연은 smk_cur·smk_former 두 더미로 한 변수)가 최소충분보정집합이며, 하나라도 빼면 열린 백도어가 남습니다. bmi는 음주가 체중을 거쳐 혈압에 미치는 경로 위의 매개변수라 보정하면 총효과 일부가 사라지므로 제외하였습니다. htn_med는 혈압의 결과(혈압 → 약 복용)라서 보정하면 결과를 일부 통제하는 셈이 되어 왜곡되므로 제외하였습니다. height는 무관해서 제외하였습니다.

> 화살표 방향이 불확실한 두 곳은 두 방향을 모두 따져 보았습니다. (1) 흡연 ↔ 음주: smk → drink이면 흡연은 교란이라 보정하고(주 모형), drink → smk이면 흡연은 매개라 빼야 하며 이때 보정집합은 {age, male, edu, income}입니다. (2) 운동 ↔ 음주: exercise → drink이면 운동은 교란이라 보정집합에 추가하고, drink → exercise이면 매개라 제외합니다(주 모형). 두 가정이 결과를 바꾸는지는 아래 ④-2의 민감도 분석에서 확인합니다.

> 한계: 식습관·염분(diet)과 과거 혈압(sbp_prev)은 자료에 없어 측정되지 않은 교란으로 남습니다. 특히 고혈압 진단 후 술을 줄이는 역방향 인과가 있으면 과거 혈압이 음주와 현재 혈압의 공통 원인이 되는데 단면자료에서는 보정할 수 없으므로, 잔여 교란 가능성을 결과 해석의 한계로 둡니다.

In [ ]:
# 개인 과제 ④ — 복합표본 회귀 (10점)
#   무보정 모형과 보정 모형을 나란히, 설계 기반(샌드위치) SE로 OR(연속형 결과면 계수)과 95% CI를 제시
#   결과가 연속형이면 family='gaussian'
#   svyglm은 결측값을 처리하지 않습니다 — 결측이 있는 행을 빼야 한다면 몇 명이 왜 빠졌는지 적으십시오

df['drink_mod']  = (df['drink'] == 2).astype(float)   # 중등도 (기준: 비음주)
df['drink_high'] = (df['drink'] == 3).astype(float)   # 고위험
df['smk_former'] = (df['smk'] == 2).astype(float)     # 과거흡연 (기준: 비흡연)

# edu·income의 9 = 무응답 → 제외. 행을 지우지 않고 가중치를 0으로 만듦 (도메인 방식, 설계 정보 유지)
ok = ((df['edu'] != 9) & (df['income'] != 9)).values
wd = w * ok
print(f"제외 {(~ok).sum()}명 (edu 무응답 {(df.edu==9).sum()}, income 무응답 {(df.income==9).sum()}) / 분석 {ok.sum():,}명")

EXP = ['drink_mod', 'drink_high']
ADJ = ['age', 'male', 'smk_cur', 'smk_former', 'edu', 'income']

P = {}
for key, cols in [('무보정', EXP), ('보정', EXP + ADJ)]:
    X = np.column_stack([np.ones(len(df)), df[cols].values])
    b, se, m = svyglm(df['sbp'], X, wd, ST, PS, family='gaussian')
    P[key] = {'cols': cols, 'b': b, 'se': se}
    for i, nm in [(1, '중등도'), (2, '고위험')]:
        print(f"{key:<4} {nm}: {b[i]:6.2f} mmHg (95% CI {b[i]-1.96*se[i]:.2f} ~ {b[i]+1.96*se[i]:.2f})")

제외 405명 (edu 무응답 80, income 무응답 326) / 분석 4,962명
무보정  중등도:  -0.53 mmHg (95% CI -1.68 ~ 0.62)
무보정  고위험:   1.84 mmHg (95% CI 0.44 ~ 3.24)
보정   중등도:  -0.34 mmHg (95% CI -1.35 ~ 0.68)
보정   고위험:   2.07 mmHg (95% CI 0.83 ~ 3.31)


In [ ]:
# 개인 과제 ④-2 — 과보정 시연 + 민감도 분석 (④의 보정 모형과 나란히)
df['exer'] = (df['exercise'] == 2).astype(float)      # 규칙적 운동 (1=아니오, 2=예)

VARIANTS = [('보정 (주 모형)',                    EXP + ADJ),
            ('+ bmi (과보정: 매개변수)',            EXP + ADJ + ['bmi']),
            ('+ htn_med (과보정: 결과의 하류)',     EXP + ADJ + ['htn_med']),
            ('− 흡연 (민감도: 흡연이 매개라면)',    EXP + ['age', 'male', 'edu', 'income']),
            ('+ exercise (민감도: 운동이 교란이라면)', EXP + ADJ + ['exer'])]

rows = []
for nm, cols in VARIANTS:
    X = np.column_stack([np.ones(len(df)), df[cols].values])
    b, se, _ = svyglm(df['sbp'], X, wd, ST, PS, family='gaussian')
    rows.append([nm] + [f"{b[i]:.2f} ({b[i]-1.96*se[i]:.2f} ~ {b[i]+1.96*se[i]:.2f})" for i in (1, 2)])
print(pd.DataFrame(rows, columns=['모형', '중등도 β (95% CI)', '고위험 β (95% CI)']).to_string(index=False))

# 변화의 이유를 확인하는 숫자 — 매개 경로(음주 → bmi → sbp)의 두 고리, 약 복용의 두 고리
Xa = np.column_stack([np.ones(len(df)), df[EXP + ADJ].values])
b1, _, _ = svyglm(df['bmi'], Xa, wd, ST, PS, family='gaussian')
print(f"\n음주 → bmi (보정 후): 중등도 {b1[1]:.2f}, 고위험 {b1[2]:.2f} kg/m²")
Xb = np.column_stack([np.ones(len(df)), df[EXP + ADJ + ['bmi']].values])
b2, _, _ = svyglm(df['sbp'], Xb, wd, ST, PS, family='gaussian')
print(f"bmi → sbp: 1 kg/m² 당 {b2[-1]:.2f} mmHg")
b3, _, _ = svyglm(df['htn_med'], Xa, wd, ST, PS, family='gaussian')
print(f"음주 → 약 복용 확률 차이 (보정 후): 중등도 {b3[1]*100:.1f}%p, 고위험 {b3[2]*100:.1f}%p")
mk = ok & (df['htn_med'] == 1).values; nk = ok & (df['htn_med'] == 0).values
print(f"평균 sbp: 약 복용 {np.average(df.loc[mk, 'sbp'], weights=w[mk]):.1f} / 비복용 {np.average(df.loc[nk, 'sbp'], weights=w[nk]):.1f} mmHg")

💬 **④-2 해석 — 과보정과 민감도 분석** (추정치가 어떻게, 왜 달라졌는가)

> **+ bmi (매개변수 보정)**: 고위험 음주의 β가 2.07에서 2.26 mmHg로 달라졌습니다. 음주 → BMI → 혈압 경로를 막았기 때문입니다. 이 자료에서는 음주군의 BMI가 오히려 약간 낮고(고위험 약 −0.2 kg/m²) BMI가 오르면 혈압이 오르므로(1 kg/m² 당 약 0.85 mmHg), 이 경로는 음주의 총효과를 약 0.2 mmHg 깎고 있었습니다. 그 경로를 막으니 그만큼이 빠져 추정치가 커진 것입니다. 변화의 방향은 경로의 부호에 달려 있지만, 총효과의 일부를 제거한다는 구조는 같으므로 bmi는 보정하면 안 됩니다.

> **+ htn_med (결과의 하류 보정)**: 고위험 2.07 → 2.00, 중등도 −0.34 → −0.45로 거의 변하지 않았습니다. 혈압이 높은 사람이 약을 먹으므로(복용자 평균 sbp가 비복용자보다 약 19 mmHg 높음) htn_med는 결과의 결과라서 구조상 보정하면 안 됩니다. 다만 음주 수준에 따른 복용 확률 차이가 1%p 안팎으로 작아 음주와 htn_med의 연결이 약하기 때문에, 이 자료에서는 왜곡이 작게 나타났을 뿐입니다. 변화가 작다는 것이 보정해도 된다는 뜻은 아닙니다.

> **민감도 분석**: 흡연을 빼도(2.16) 운동을 넣어도(2.11) 고위험 음주의 β는 주 모형의 2.07과 비슷하고 신뢰구간이 크게 겹칩니다. 방향이 불확실한 두 화살표를 어느 쪽으로 가정해도 결론이 달라지지 않습니다. 결론은 고위험 음주군이 비음주군보다 수축기혈압이 약 2 mmHg 높다는 것이며, 중등도 음주는 모든 모형에서 신뢰구간이 0을 포함해 차이가 있다고 말할 근거가 없습니다.

In [ ]:
# 개인 과제 ⑤ — Table 2 (5점)
#   열: 무보정 | 보정 · 관심 노출을 맨 위 행에

fmt = lambda c, lo, hi: f"{c:.2f} ({lo:.2f} ~ {hi:.2f})"
ci  = lambda d, i: (d['b'][i], d['b'][i]-1.96*d['se'][i], d['b'][i]+1.96*d['se'][i])

rows = []
for label, v in [('중등도 음주 (vs 비음주)', 'drink_mod'), ('고위험 음주 (vs 비음주)', 'drink_high')]:
    rows.append([label, fmt(*ci(P['무보정'], P['무보정']['cols'].index(v)+1)),
                        fmt(*ci(P['보정'],   P['보정']['cols'].index(v)+1))])
for label, v in [('연령 (1세 증가)', 'age'), ('남성', 'male'), ('현재흡연 (vs 비흡연)', 'smk_cur'),
                 ('과거흡연 (vs 비흡연)', 'smk_former'), ('교육수준 (1단계 증가)', 'edu'), ('소득 (1분위 증가)', 'income')]:
    rows.append([label, '—', fmt(*ci(P['보정'], P['보정']['cols'].index(v)+1))])

table2 = pd.DataFrame(rows, columns=['변수', '무보정 β (95% CI)', '보정 β (95% CI)'])
print(table2.to_string(index=False))

             변수       무보정 β (95% CI)         보정 β (95% CI)
중등도 음주 (vs 비음주) -0.53 (-1.68 ~ 0.62)  -0.34 (-1.35 ~ 0.68)
고위험 음주 (vs 비음주)   1.84 (0.44 ~ 3.24)    2.07 (0.83 ~ 3.31)
     연령 (1세 증가)                    —    0.59 (0.56 ~ 0.62)
             남성                    — -2.75 (-3.68 ~ -1.82)
  현재흡연 (vs 비흡연)                    —    3.65 (2.46 ~ 4.85)
  과거흡연 (vs 비흡연)                    —   0.82 (-0.33 ~ 1.98)
  교육수준 (1단계 증가)                    —  -0.27 (-0.93 ~ 0.39)
    소득 (1분위 증가)                    —  -0.09 (-0.54 ~ 0.36)


💬 **⑤ Table 2 각주** — 모형 종류 · 보정변수 목록 · 설계 반영 방법(층화·집락·가중치, Taylor 선형화) · Table 2 fallacy 경고 문장

> 결과변수 = 수축기혈압(mmHg). β는 비음주군 대비 평균 차이(mmHg)이며, 설계 기반 복합표본 선형회귀로 추정하였습니다(n = 4,962). 보정 모형은 연령·성별·흡연상태·교육수준·소득을 함께 보정하였고, 교육·소득 무응답 405명은 가중치를 0으로 처리해 제외하였습니다. 통합가중치로 계수를 추정하고, 층(18개)·집락(263개 조사구)을 반영한 샌드위치(Taylor 선형화) 표준오차로 95% 신뢰구간을 계산하였습니다. 보정변수의 계수는 음주의 효과를 추정하기 위해 구성한 모형에서 나온 값이므로 인과적으로 해석할 수 없습니다.

## ⑥ Methods 문단 (개인 과제, 4~7문장)

💬 **Methods 문단**

> 음주수준(비음주 대비 중등도·고위험)이 수축기혈압에 미치는 총효과를 추정하기 위해 DAG를 그려 변수의 역할을 구분했다. age, male, 흡연상태(smk_cur·smk_former), edu, income은 음주보다 먼저 있으면서 음주와 혈압 양쪽에 영향을 주는 교란변수이므로 보정집합으로 정했고, 음주 → bmi → sbp 경로의 매개변수 bmi, 혈압의 결과인 htn_med, 무관한 height는 제외했다. 표본가중치(wt)를 반영한 선형회귀(svyglm, gaussian)로 계수와 95% CI를 구했고, 층(kstrata, 18개)과 집락(psu, 263개 조사구)을 반영한 샌드위치(Taylor 선형화) 분산추정으로 표준오차를 계산했다. 교육·소득 무응답 405명은 가중치를 0으로 처리해 제외하고 4,962명을 분석했다. 비음주 대비 고위험 음주의 수축기혈압 차이는 무보정 1.84 mmHg(0.44~3.24), 보정 2.07 mmHg(0.83~3.31)였고, 중등도 음주는 보정 후 −0.34 mmHg(−1.35~0.68)로 0을 포함했다. bmi를 추가하면 2.26 mmHg로 달라져 매개 경로를 막은 효과가 확인되었고, 흡연을 빼거나 운동을 넣어도 2.16, 2.11 mmHg로 결론은 유지되었다. 식습관과 과거 혈압은 측정되지 않아 잔여 교란이 남을 수 있다.